In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import psutil
sys.path.append("..")  # Adds higher directory to python modules path.
from src import IOFunctions
from src import HelperFunctions
IO = IOFunctions.IO_Functions()
HF = HelperFunctions.Helper_Functions()
import dask.array as da

In [ ]:
folders = ['/media/jbeckwith/Ezra Seagat/JSB/20241126/data', '/media/jbeckwith/Ezra Seagat/JSB/20241124/data', 
           '/media/jbeckwith/Ezra Seagat/JSB/20241025/data', '/media/jbeckwith/Ezra Seagat/JSB/20241016/data',
           '/media/jbeckwith/Ezra Seagat/JSB/20241010/data', '/media/jbeckwith/Ezra Seagat/JSB/20241009/data']

In [ ]:
def moving_average(a, n=5):
    ret = da.cumsum(a, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    return ret[n - 1:] / n

In [ ]:
def file_input(data, median, file):
    if data is None:
        data = da.from_array(IO.read_tiff(file, dtype=np.uint16))
    else:
        data = da.dstack([data, da.from_array(IO.read_tiff(file, dtype=np.uint16))])
    median = da.median(data, axis=[0, 1])
    return data, median

In [ ]:
def check_for_cp(data, median, min_length=100, sigma=3, ma=5):
    one_d = da.expand_dims(moving_average(median, n=ma), axis=1)
    cp_coords = np.asarray(da.where(da.abs(one_d-da.median(one_d, axis=0)) > sigma*da.std(one_d, axis=0))[0])
    aboveT_cp = np.diff(cp_coords) > min_length
    if sum(aboveT_cp) > 0:
        flag = True
        cutting_points = cp_coords[np.hstack([True, aboveT_cp])]
        if da.abs(one_d[0]-da.median(one_d, axis=0)) < sigma*da.std(one_d, axis=0):
            return flag, np.hstack([0, cutting_points])
        else:
            return flag, cutting_points
    else:
        flag = False
        cutting_points = None
        return flag, cutting_points

In [ ]:
def save_chunk(data, median, unique_file, cutting_points, p):
    pos_string = '_Position_'+str(p).zfill(4)+'.tif'
    file_path = unique_file+pos_string
    IO.write_tiff(data[:, :, cutting_points[0]:cutting_points[1]], file_path)
    print("written "+os.path.split(file_path)[-1], end="\r", flush=True)
    p += 1
    data = data[:, :, cutting_points[1]:]
    median = median[cutting_points[1]:]
    return data, median, p

In [ ]:
def save_final(data, unique_file, cutting_points, p):
    for i in np.arange(len(cutting_points)-1):
        pos_string = '_Position_'+str(p).zfill(4)+'.tif'
        file_path = unique_file+pos_string
        IO.write_tiff(data[:, :, cutting_points[i]:cutting_points[i+1]], file_path)
        print("written "+os.path.split(file_path)[-1], end="\r", flush=True)
        p += 1
    return

In [ ]:
def thorlabs_file_chunker(folder, min_time=10, sigma=3):
    files = np.sort(HF.file_search(folder, '.tif', ''))
    unique_files = np.unique(['_'.join(x.split('.tif')[0].split('_')[:-1]) for x in files])
    for unique_file in unique_files:
        if '_20ms_' in unique_file:
            min_length = int(min_time/0.02)
        else:
            min_length = int(min_time/0.1)
        ma = 5
        os.system('clear')
        print("saving "+os.path.split(unique_file)[-1])
        p = 0
        file_counter = 0
        file_pertaining = np.sort([x for x in files if unique_file in x])
        data = None
        median = None
        flag = False
        while file_counter < len(file_pertaining)-1:
            data, median = file_input(data, median, file_pertaining[file_counter])
            file_counter += 1
            flag, cutting_points = check_for_cp(data, median, min_length, sigma, ma)
            if flag == False:
                continue
            else:
                print(cutting_points)
                data, median, p = save_chunk(data, median, unique_file, cutting_points, p)
        if data.shape[-1] > min_length:
            flag, cutting_points = check_for_cp(data, median, min_length, sigma)
            save_final(data, unique_file, cutting_points, p)
        del data
    return

In [ ]:
for folder in folders:
    thorlabs_file_chunker(folder, min_time=2, sigma=4)